# EDA: retención de grupos escolares en Scholastic Travel Company

**Contexto:** proyecto académico basado en el caso *Retention Modeling at Scholastic Travel Company (A)* (Anton S. Ovchinnikov, Darden School of Business, 2017), que plantea predecir la retención de clientes a partir de sus atributos.

**Datos:** 2.389 grupos escolares y 56 variables (programa, grados, tipo de colegio, fechas de salida, precio, participantes, reuniones con padres y retención). Los datos pertenecen al caso y no se incluyen en este repositorio: para ejecutar el notebook, el CSV del caso debe estar en la misma carpeta.
**Variable objetivo:** `Retained.in.2012.` (binaria).

**Qué hago en este notebook**
1. Cargo los datos y reviso su estructura
2. Diagnostico los valores faltantes (¿son aleatorios?)
3. Limpio y transformo sin inventar datos
4. Valido la consistencia
5. Analizo las variables una por una
6. Cuantifico los outliers
7. Estudio qué se relaciona con la retención
8. Genero un reporte automático y exporto el dataset curado

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.3f}".format)

RUTA_DATOS = "03 CSV data -- STC(A)_numerical dates.csv"
OBJETIVO = "Retained.in.2012."

os.makedirs("imagenes", exist_ok=True)

## 1. Carga y vista general

In [ ]:
df_raw = pd.read_csv(RUTA_DATOS)
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")
print("\nTipos de datos:")
print(df_raw.dtypes.astype(str).value_counts().to_string())
df_raw.head(3).T

## 2. Calidad de datos: valores faltantes

In [ ]:
faltantes = (
    df_raw.isnull().sum().to_frame("n_nulos")
    .assign(pct=lambda t: t["n_nulos"] / len(df_raw) * 100)
    .query("n_nulos > 0")
    .sort_values("n_nulos", ascending=False)
)
display(faltantes)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(x=faltantes["pct"], y=faltantes.index, color="#3b82f6", ax=ax)
ax.set(xlabel="% de nulos", ylabel="", title="Valores faltantes por columna")
plt.tight_layout()
fig.savefig("imagenes/valores_faltantes.png", dpi=130, bbox_inches="tight")
plt.show()

print(f"IDs únicos: {df_raw['ID'].is_unique} | Filas duplicadas: {df_raw.duplicated().sum()}")

### ¿Los nulos son aleatorios?

Antes de imputar necesito entender **por qué** faltan los datos. `FirstMeeting`, `LastMeeting` y las dos diferencias de días hacia las reuniones tienen exactamente el mismo número de nulos, así que las contrasto con el número de reuniones con padres:

In [ ]:
sin_reunion = df_raw["FirstMeeting"].isnull().rename("FirstMeeting es nulo")
display(
    df_raw.groupby(sin_reunion)["NumberOfMeetingswithParents"]
    .agg(grupos="size", minimo="min", promedio="mean", maximo="max")
)

**Lectura:** los 337 nulos de `FirstMeeting`, `LastMeeting` y de las dos diferencias de días corresponden **exactamente** a los grupos con **0 reuniones con padres**; todos los demás tienen 1 o 2. Es un nulo **estructural** (el evento no ocurrió), no un dato perdido. Si lo imputara con la mediana fabricaría fechas de reunión que nunca existieron, así que lo conservo como nulo.

Los otros nulos relevantes: `Special.Pay` (80 %, la elimino), `Early.RPL` (28 %) y `Poverty.Code` (25 %).

## 3. Limpieza y transformación

Mis decisiones:
- Elimino las columnas con **más de 50 % de nulos**.
- Convierto a fecha las **8 columnas** en formato serial de Excel.
- No imputo el nulo estructural de reuniones; el dataset ya trae `Parent.Meeting.Flag`, que resume si hubo reuniones.
- En las categóricas, el nulo pasa a ser una categoría explícita (`Desconocido`) en lugar de reemplazarlo por la moda.
- No imputo los nulos numéricos restantes en el EDA: lo haría dentro de un pipeline de modelado, ajustado solo con datos de entrenamiento, para evitar fuga de información.

In [ ]:
df = df_raw.copy()

# 3.1 Columnas con más de 50 % de nulos
cols_muy_vacias = df.columns[df.isnull().mean() > 0.5].tolist()
df = df.drop(columns=cols_muy_vacias)
print("Columnas eliminadas por >50 % de nulos:", cols_muy_vacias)

# 3.2 Fechas seriales de Excel -> datetime (los nulos pasan a NaT)
cols_fecha = ["Departure.Date", "Return.Date", "Deposit.Date", "Initial.System.Date",
              "Early.RPL", "Latest.RPL", "FirstMeeting", "LastMeeting"]
for c in cols_fecha:
    df[c] = pd.to_datetime(df[c], unit="D", origin="1899-12-30")

# 3.3 Categóricas: el nulo se vuelve una categoría explícita
cols_cat = ["Program.Code", "Group.State", "Travel.Type", "Region", "School.Type",
            "SPR.Product.Type", "SPR.New.Existing", "Poverty.Code", "Income.Level",
            "MDR.Low.Grade", "SchoolSizeIndicator", "MajorProgramCode", "DepartureMonth"]
for c in cols_cat:
    df[c] = df[c].astype("object").fillna("Desconocido").astype(str).astype("category")

# 3.4 Nulos que quedan sin imputar
restantes = df.isnull().sum()
display(restantes[restantes > 0].to_frame("nulos_restantes"))

## 4. Validación de consistencia

Reporto las inconsistencias en lugar de borrar filas sin revisarlas.

In [ ]:
ambos = df["From.Grade"].notna() & df["To.Grade"].notna()
chequeos = {
    "Filas duplicadas": int(df.duplicated().sum()),
    "From.Grade > To.Grade (donde ambos existen)": int((ambos & (df["From.Grade"] > df["To.Grade"])).sum()),
    "Departure.Date > Return.Date": int((df["Departure.Date"] > df["Return.Date"]).sum()),
    "Deposit.Date > Departure.Date": int((df["Deposit.Date"] > df["Departure.Date"]).sum()),
    "Days no coincide con (Return - Departure + 1)": int((df["Days"] != (df["Return.Date"] - df["Departure.Date"]).dt.days + 1).sum()),
    "Tuition distinta de SPR.Group.Revenue": int((df["Tuition"] != df["SPR.Group.Revenue"]).sum()),
    "Parent.Meeting.Flag inconsistente con NumberOfMeetingswithParents": int((df["Parent.Meeting.Flag"] != (df["NumberOfMeetingswithParents"] > 0).astype(int)).sum()),
    "Total.Pax distinto de FPP + Num.of.Non_FPP.PAX": int((df["Total.Pax"] != df["FPP"] + df["Num.of.Non_FPP.PAX"]).sum()),
    "Num.of.Non_FPP.PAX distinta de Total.Discount.Pax": int((df["Num.of.Non_FPP.PAX"] != df["Total.Discount.Pax"]).sum()),
}
display(pd.Series(chequeos, name="n_filas").to_frame())
print(f"Días que no coinciden con las fechas: {chequeos['Days no coincide con (Return - Departure + 1)'] / len(df):.1%} de los grupos")
print("\nGrupos con depósito posterior a la salida:")
display(df.loc[df["Deposit.Date"] > df["Departure.Date"], ["ID", "Deposit.Date", "Departure.Date"]])

**Hallazgos**
- No hay duplicados, los IDs son únicos, no hay grupos con `From.Grade > To.Grade` ni salidas posteriores al regreso.
- El grupo con **ID 1575** tiene el depósito en octubre de 2011 y la salida en junio de 2011: parece un error de captura. Lo conservo y lo dejo señalado.
- En **322 grupos (13,5 %)** `Days` no coincide con (regreso − salida + 1). Hay que confirmar qué cuenta exactamente `Days`.
- `Tuition` y `SPR.Group.Revenue` son **idénticas en el 100 % de las filas**: una de las dos es redundante.
- `Parent.Meeting.Flag` coincide con `NumberOfMeetingswithParents > 0` en todas las filas.
- `Total.Pax` es igual a `FPP` + `Num.of.Non_FPP.PAX` en todas las filas, y `Num.of.Non_FPP.PAX` es idéntica a `Total.Discount.Pax`: son variables redundantes.

## 5. Análisis univariado

In [ ]:
cols_num_clave = ["Tuition", "Days", "Total.Pax", "SPR.Group.Revenue", "FPP",
                  "FRP.Take.up.percent.", "Total.School.Enrollment"]
display(df[cols_num_clave].describe().T.assign(asimetria=df[cols_num_clave].skew()))

fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for ax, col in zip(axes.ravel()[:4], ["Tuition", "Days", "Total.Pax", "SPR.Group.Revenue"]):
    sns.histplot(df[col], bins=30, kde=True, color="#3b82f6", ax=ax)
    ax.set_title(f"Distribución de {col}")
sns.countplot(data=df, x="Travel.Type", color="#3b82f6", ax=axes[1, 1])
axes[1, 1].set_title("Frecuencia de Travel.Type")
orden_region = df["Region"].value_counts().index
sns.countplot(data=df, x="Region", order=orden_region, color="#3b82f6", ax=axes[1, 2])
axes[1, 2].set_title("Frecuencia de Region")
axes[1, 2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

display(df["Program.Code"].value_counts(normalize=True).head(8).to_frame("proporcion"))

**Hallazgos**
- `Tuition` es casi simétrica (media 1.615, mediana 1.700, asimetría −0,13).
- `Total.Pax` (mediana 26, máximo 313), `FPP` y `Total.School.Enrollment` están muy sesgadas a la derecha (asimetrías 2,9, 2,8 y 2,2): conviene una transformación logarítmica al modelar.
- `Days`: mediana de 5 días, con la mayoría de los grupos entre 4 y 5.
- Un programa domina: `HD` concentra el 59,9 % de los grupos, y `Travel.Type = A` el 84 %.
- `FRP.Take.up.percent.` tiene mediana 0,60 y sesgo negativo (−0,70).

## 6. Outliers (regla del rango intercuartílico)

Cuantifico los outliers pero no los elimino: en estos datos un grupo muy grande o un programa caro puede ser legítimo.

In [ ]:
def resumen_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = int(((serie < lim_inf) | (serie > lim_sup)).sum())
    return pd.Series({"q1": q1, "q3": q3, "lim_inf": lim_inf, "lim_sup": lim_sup,
                      "n_outliers": n, "pct_outliers": n / len(serie) * 100})

cols_outlier = ["Tuition", "Days", "Total.Pax", "SPR.Group.Revenue", "FPP"]
display(pd.DataFrame({c: resumen_iqr(df[c].dropna()) for c in cols_outlier}).T)

top_prog = df["Program.Code"].value_counts().head(8).index
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df[df["Program.Code"].isin(top_prog)], x="Program.Code", y="Tuition",
            order=top_prog, color="#93c5fd", ax=ax)
ax.set_title("Tuition por Program.Code (8 programas más frecuentes)")
plt.tight_layout()
plt.show()

**Hallazgos**
- `Tuition`: 15 outliers (0,6 %) sobre 3.359. `Total.Pax`: 117 (4,9 %) sobre 89. `FPP`: 111 (4,6 %).
- `Days` marca 328 (13,7 %), pero solo porque casi todos los grupos están en 4–5 días y el rango intercuartílico es estrecho: no son errores.

**Decisión:** no los elimino. Los grupos grandes son legítimos y, como veo más abajo, el tamaño del grupo se asocia con la retención.

## 7. Relación con la variable objetivo

`Retained.in.2012.` es binaria; asumo **1 = retenido en 2012** (hay que confirmarlo con el diccionario de datos del caso). Comparo la tasa de retención entre categorías y mido la fuerza de la asociación con la **V de Cramér** (0 = sin relación, 1 = relación perfecta).

In [ ]:
tasa_global = df[OBJETIVO].mean()
print(f"Tasa de retención global: {tasa_global:.1%} (n = {len(df):,})")

vars_cat = ["Is.Non.Annual.", "SingleGradeTripFlag", "Parent.Meeting.Flag", "SPR.New.Existing",
            "Travel.Type", "School.Type", "Region", "SPR.Product.Type"]

def tasa_por(col, min_n=30):
    return (df.groupby(col, observed=True)[OBJETIVO].agg(tasa="mean", n="size")
              .query("n >= @min_n").sort_values("tasa", ascending=False))

fig, axes = plt.subplots(2, 4, figsize=(19, 8))
for ax, col in zip(axes.ravel(), vars_cat):
    t = tasa_por(col)
    sns.barplot(x=t.index.astype(str), y=t["tasa"], color="#3b82f6", ax=ax)
    ax.axhline(tasa_global, color="red", ls="--", lw=1)
    ax.set_title(col)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
    for i, n in enumerate(t["n"]):
        ax.text(i, t["tasa"].iloc[i] + 0.02, f"n={n}", ha="center", fontsize=8)
fig.suptitle("Tasa de retención por categoría (línea roja = tasa global; solo categorías con n ≥ 30)", y=1.02)
plt.tight_layout()
fig.savefig("imagenes/retencion_por_categoria.png", dpi=130, bbox_inches="tight")
plt.show()

filas = []
for col in vars_cat:
    tabla = pd.crosstab(df[col], df[OBJETIVO])
    chi2, p, _, _ = stats.chi2_contingency(tabla)
    v = np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))
    filas.append({"variable": col, "chi2": chi2, "p_valor": p, "V_Cramer": v})
display(pd.DataFrame(filas).sort_values("V_Cramer", ascending=False).reset_index(drop=True))

**Hallazgos** (tasa global de retención: **60,7 %**)
- `SingleGradeTripFlag`: 80,6 % (n = 1.330) frente a 35,8 % (n = 1.059). V de Cramér 0,46: la asociación más fuerte.
- `Is.Non.Annual.`: 16,0 % (n = 368) frente a 68,9 % (n = 2.021). V = 0,39.
- `SPR.New.Existing`: clientes existentes 73,3 % frente a nuevos 34,9 %. V = 0,37.
- Efecto combinado: existente + un solo grado → 85,5 % (n = 1.072); nuevo + varios grados → 22,5 % (n = 524).
- `Region` y `SPR.Product.Type` tienen asociación débil o moderada (V = 0,12 y 0,16). Costa Rica (26,1 %, n = 46) e International (6,7 %, n = 15) retienen poco, pero con muy pocos grupos: interpretar con cautela.
- `Parent.Meeting.Flag` casi no se relaciona con la retención (60,3 % con reuniones frente a 63,2 % sin; V = 0,02; p = 0,35).

In [ ]:
cols_excluir = ["ID", "CRM.Segment", OBJETIVO]
num_cols = df.select_dtypes("number").columns.difference(cols_excluir)
corr_obj = df[num_cols].corrwith(df[OBJETIVO]).dropna().sort_values(key=abs, ascending=False)
top = corr_obj.head(12)

fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={"width_ratios": [1, 1.2]})
sns.barplot(x=top.values, y=top.index, color="#3b82f6", ax=axes[0])
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_ylabel("")
axes[0].set_title("Correlación con la retención (12 variables numéricas principales)")
top10 = list(top.index[:10]) + [OBJETIVO]
sns.heatmap(df[top10].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1], cbar=False)
axes[1].set_title("Matriz de correlación (top 10 + objetivo)")
plt.tight_layout()
fig.savefig("imagenes/correlacion_retencion.png", dpi=130, bbox_inches="tight")
plt.show()

**Hallazgos**
- Mayor correlación con la retención: `SingleGradeTripFlag` (+0,46), `Is.Non.Annual.` (−0,39), `FPP` (+0,26), `Total.Pax` (+0,26) y `FRP.Active` (+0,25).
- La retención crece con el tamaño del grupo: **40,4 %** en el cuartil más pequeño de `Total.Pax` frente a **80,5 %** en el más grande.

**Precauciones**
- Correlación no es causalidad.
- `FPP`, `Total.Pax`, `FRP.Active`, `Num.of.Non_FPP.PAX` y `Total.Discount.Pax` están muy correlacionadas entre sí (de 0,70 a 1,00): hay multicolinealidad que atender antes de modelar.
- Algunas variables (cancelaciones, ingresos finales) se conocen **después** del viaje. Si el objetivo es predecir la retención antes o durante la salida, hay que revisar cuáles estarían disponibles en ese momento (riesgo de fuga de información).

## 8. Reporte automático y exportación

Genero el reporte de ydata-profiling en modo `minimal=True`, que es liviano y se crea en segundos. Si quiero el reporte completo (correlaciones e interacciones), cambio a `minimal=False`.

Los tres gráficos principales (valores faltantes, retención por categoría y correlaciones) se guardan además como imágenes en la carpeta `imagenes/`, que uso en el README.

In [ ]:
GENERAR_REPORTE = True

if GENERAR_REPORTE:
    try:
        from ydata_profiling import ProfileReport
    except ImportError:
        from data_profiling import ProfileReport
    perfil = ProfileReport(df, title="Reporte EDA", minimal=True)
    perfil.to_file("eda_report.html")
    print("Reporte guardado en eda_report.html")

df.to_csv("data_curada.csv", index=False)
print(f"Dataset curado exportado: data_curada.csv ({df.shape[0]:,} filas, {df.shape[1]} columnas)")

## 9. Conclusiones

- Casi 6 de cada 10 grupos (60,7 %) fueron retenidos en 2012.
- La retención se asocia sobre todo con **tres factores**: viajes de un solo grado, programas anuales (frente a no anuales) y clientes existentes (frente a nuevos). Combinados, la tasa va de 22,5 % a 85,5 %.
- Los grupos más grandes se retienen más.
- Las reuniones con padres casi no muestran relación con la retención.
- Son asociaciones descriptivas, no causales.